In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All"
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/3d-ircad20/Dataset001_Liver/dataset.json
/kaggle/input/3d-ircad20/Dataset001_Liver/labelsTr/liver_010.nii
/kaggle/input/3d-ircad20/Dataset001_Liver/labelsTr/liver_005.nii
/kaggle/input/3d-ircad20/Dataset001_Liver/labelsTr/liver_003.nii
/kaggle/input/3d-ircad20/Dataset001_Liver/labelsTr/liver_015.nii
/kaggle/input/3d-ircad20/Dataset001_Liver/labelsTr/liver_004.nii
/kaggle/input/3d-ircad20/Dataset001_Liver/labelsTr/liver_016.nii
/kaggle/input/3d-ircad20/Dataset001_Liver/labelsTr/liver_008.nii
/kaggle/input/3d-ircad20/Dataset001_Liver/labelsTr/liver_013.nii
/kaggle/input/3d-ircad20/Dataset001_Liver/labelsTr/liver_006.nii
/kaggle/input/3d-ircad20/Dataset001_Liver/labelsTr/liver_001.nii
/kaggle/input/3d-ircad20/Dataset001_Liver/labelsTr/liver_014.nii
/kaggle/input/3d-ircad20/Dataset001_Liver/labelsTr/liver_000.nii
/kaggle/input/3d-ircad20/Dataset001_Liver/labelsTr/liver_002.nii
/kaggle/input/3d-ircad20/Dataset001_Liver/labelsTr/liver_017.nii
/kaggle/input/3d-ircad20/Dataset001

In [ ]:
import nibabel as nib
import numpy as np
import os

IMG_NII = "/kaggle/input/3d-ircad20/Dataset001_Liver/imagesTr"
MSK_NII = "/kaggle/input/3d-ircad20/Dataset001_Liver/labelsTr"

img_file = sorted(os.listdir(IMG_NII))[0]
patient_id = img_file.split("_")[1]
mask_file = f"liver_{patient_id}.nii"

img = nib.load(os.path.join(IMG_NII, img_file)).get_fdata()
msk = nib.load(os.path.join(MSK_NII, mask_file)).get_fdata()

print("Image shape:", img.shape)
print("Mask shape :", msk.shape)
print("Mask unique values:", np.unique(msk))
print("Foreground voxels:", np.sum(msk > 0))


Image shape: (512, 512, 129)
Mask shape : (512, 512, 129)
Mask unique values: [0. 1.]
Foreground voxels: 2865131


In [ ]:
# counts = {
#     "axis0": 0,
#     "axis1": 0,
#     "axis2": 0
# }

# for z in range(msk.shape[0]):
#     if np.sum(msk[z, :, :]) > 0:
#         counts["axis0"] += 1

# for z in range(msk.shape[1]):
#     if np.sum(msk[:, z, :]) > 0:
#         counts["axis1"] += 1

# for z in range(msk.shape[2]):
#     if np.sum(msk[:, :, z]) > 0:
#         counts["axis2"] += 1

# print(counts)


In [ ]:
# import cv2

# OUT = "/kaggle/working/debug_slices"
# os.makedirs(OUT, exist_ok=True)

# saved = 0

# for z in range(msk.shape[0]):  # CHANGE AXIS IF NEEDED
#     if np.sum(msk[z]) > 0:
#         img_slice = img[z]
#         img_slice = cv2.normalize(img_slice, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)
#         cv2.imwrite(f"{OUT}/slice_{z}.png", img_slice)
#         saved += 1

#     if saved == 5:
#         break

# print("Saved slices:", saved)


# **converting data into Ymal format**

In [ ]:
# # remove the directories
# !rm -rf /kaggle/working/rf-detr-base.pth

In [ ]:
import os
import cv2
import yaml
import random
import numpy as np
import nibabel as nib
from tqdm import tqdm

# ===============================
# INPUT PATHS
# ===============================
IMG_NII = "/kaggle/input/3d-ircad20/Dataset001_Liver/imagesTr"
MSK_NII = "/kaggle/input/3d-ircad20/Dataset001_Liver/labelsTr"

# ===============================
# OUTPUT PATHS
# ===============================
YOLO_ROOT = "/kaggle/working/yolo_dataset"
IMG_TRAIN = f"{YOLO_ROOT}/images/train"
IMG_VAL   = f"{YOLO_ROOT}/images/val"
LBL_TRAIN = f"{YOLO_ROOT}/labels/train"
LBL_VAL   = f"{YOLO_ROOT}/labels/val"

for p in [IMG_TRAIN, IMG_VAL, LBL_TRAIN, LBL_VAL]:
    os.makedirs(p, exist_ok=True)

# ===============================
# PARAMETERS
# ===============================
VAL_RATIO = 0.2
CLASS_ID = 0

# ===============================
# IMAGE FILES
# ===============================
image_files = sorted(
    [f for f in os.listdir(IMG_NII) if f.endswith(".nii")]
)

random.shuffle(image_files)

split_idx = int(len(image_files) * (1 - VAL_RATIO))
train_files = image_files[:split_idx]
val_files   = image_files[split_idx:]

print(f"Train patients: {len(train_files)} | Val patients: {len(val_files)}")

# ===============================
# MASK → YOLO SEG
# ===============================
def mask_to_yolo(mask):
    h, w = mask.shape
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    labels = []

    for cnt in contours:
        if len(cnt) < 3:
            continue

        cnt = cnt.squeeze().astype(float)
        cnt[:, 0] /= w
        cnt[:, 1] /= h

        labels.append(
            " ".join([str(CLASS_ID)] + [f"{v:.6f}" for v in cnt.flatten()])
        )

    return labels

# ===============================
# PROCESS
# ===============================
def process(files, img_out, lbl_out):
    for img_fname in tqdm(files):
        # liver_000_0000.nii → 000
        patient_id = img_fname.split("_")[1]

        img_path  = os.path.join(IMG_NII, img_fname)
        mask_path = os.path.join(MSK_NII, f"liver_{patient_id}.nii")

        img = nib.load(img_path).get_fdata()
        msk = nib.load(mask_path).get_fdata()

        # mask already 0/1
        msk = msk.astype(np.uint8)

        # ✅ CORRECT AXIS: Z = axis 2
        for z in range(img.shape[2]):
            mask_slice = msk[:, :, z]
            if mask_slice.sum() == 0:
                continue

            img_slice = img[:, :, z]
            img_slice = cv2.normalize(
                img_slice, None, 0, 255, cv2.NORM_MINMAX
            ).astype(np.uint8)

            img_name = f"liver_{patient_id}_z{z}.png"
            lbl_name = img_name.replace(".png", ".txt")

            cv2.imwrite(os.path.join(img_out, img_name), img_slice)

            labels = mask_to_yolo(mask_slice * 255)
            with open(os.path.join(lbl_out, lbl_name), "w") as f:
                for line in labels:
                    f.write(line + "\n")

# ===============================
# RUN
# ===============================
process(train_files, IMG_TRAIN, LBL_TRAIN)
process(val_files, IMG_VAL, LBL_VAL)

# ===============================
# WRITE data.yaml
# ===============================
yaml_data = {
    "path": YOLO_ROOT,
    "train": "images/train",
    "val": "images/val",
    "names": {0: "organ"}
}

with open(f"{YOLO_ROOT}/data.yaml", "w") as f:
    yaml.dump(yaml_data, f)

print("✅ YOLO dataset created successfully")


Train patients: 16 | Val patients: 4


100%|██████████| 4/4 [00:07<00:00,  1.78s/it]

✅ YOLO dataset created successfully


In [ ]:
# import os

# ROOT = "/kaggle/working/yolo_dataset"

# paths = [
#     "images/train",
#     "images/val",
#     "labels/train",
#     "labels/val",
#     "data.yaml"
# ]

# print("📂 Checking directory structure:\n")
# for p in paths:
#     full = os.path.join(ROOT, p)
#     print(f"{p}: {'✅ EXISTS' if os.path.exists(full) else '❌ MISSING'}")


In [ ]:
# def count_files(img_dir, lbl_dir):
#     imgs = sorted([f for f in os.listdir(img_dir) if f.endswith(".png")])
#     lbls = sorted([f for f in os.listdir(lbl_dir) if f.endswith(".txt")])

#     print(f"Images: {len(imgs)}")
#     print(f"Labels: {len(lbls)}")

#     missing = set(imgs) - set(l.replace(".txt", ".png") for l in lbls)
#     if missing:
#         print("❌ Images without labels:", list(missing)[:5])
#     else:
#         print("✅ All images have labels")

# print("\n🔍 TRAIN SET")
# count_files(
#     f"{ROOT}/images/train",
#     f"{ROOT}/labels/train"
# )

# print("\n🔍 VAL SET")
# count_files(
#     f"{ROOT}/images/val",
#     f"{ROOT}/labels/val"
# )


# **Distialtion**

In [ ]:
# !pip install "lightly-train[rtdetr]" "supervision==0.25.1"
!git clone https://github.com/lyuwenyu/RT-DETR.git

Cloning into 'RT-DETR'...
remote: Enumerating objects: 1119, done.
remote: Counting objects: 100% (33/33), done.
remote: Compressing objects: 100% (28/28), done.
remote: Total 1119 (delta 13), reused 5 (delta 5), pack-reused 1086 (from 3)
Receiving objects: 100% (1119/1119), 669.11 KiB | 15.93 MiB/s, done.
Resolving deltas: 100% (527/527), done.


In [ ]:
%cd /kaggle/working/RT-DETR/rtdetrv2_pytorch
!pip install lightly-train -r requirements.txt

/kaggle/working/RT-DETR/rtdetrv2_pytorch
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 492.7/492.7 kB 18.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 585.6/585.6 kB 28.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 300.5/300.5 MB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 859.3/859.3 kB 42.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.0/46.0 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 165.6/165.6 kB 11.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.5/154.5 kB 11.4 MB/s eta 0:00:00


In [ ]:
import os
output_path = "/kaggle/working/distilation_RTDTER_results"
os.makedirs(output_path, exist_ok=True)
print(f"{p}: {'✅ EXISTS' if os.path.exists(output_path) else '❌ MISSING'}")

/kaggle/working/yolo_dataset/labels/val: ✅ EXISTS


In [ ]:
from src.core import YAMLConfig

import lightly_train
from lightly_train.model_wrappers import RTDETRModelWrapper
config = YAMLConfig("configs/rtdetr/rtdetr_r50vd_6x_coco.yml")
model = config.model


2025-12-27 15:06:11.837633: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1766847972.032932      24 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1766847972.089284      24 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1766847972.537149      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1766847972.537195      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1766847972.537197      24 computation_placer.cc:177] computation placer alr

Downloading: "https://github.com/lyuwenyu/storage/releases/download/v0.1/ResNet50_vd_ssld_v2_pretrained_from_paddle.pth" to /root/.cache/torch/hub/checkpoints/ResNet50_vd_ssld_v2_pretrained_from_paddle.pth


100%|██████████| 90.0M/90.0M [00:00<00:00, 375MB/s]


Load PResNet50 state_dict


In [ ]:
import lightly_train

data_path = "/kaggle/working/yolo_dataset/images"
if __name__ == "__main__":
    wrapped_model = RTDETRModelWrapper(model)
    lightly_train.pretrain(
        out=output_path,
        data=data_path,
        model=wrapped_model,
        method="distillationv1",
        overwrite=True,
        method_args={
            "teacher": "dinov3/vits16",
            "teacher_url":"https://dinov3.llamameta.net/dinov3_vits16/dinov3_vits16_pretrain_lvd1689m-08c60483.pth?Policy=eyJTdGF0ZW1lbnQiOlt7InVuaXF1ZV9oYXNoIjoidHZpMngydGx1cjN0eHNtMWd1YWdpYjF3IiwiUmVzb3VyY2UiOiJodHRwczpcL1wvZGlub3YzLmxsYW1hbWV0YS5uZXRcLyoiLCJDb25kaXRpb24iOnsiRGF0ZUxlc3NUaGFuIjp7IkFXUzpFcG9jaFRpbWUiOjE3NTU0OTM0MDZ9fX1dfQ__&Signature=MojlptxwiEPfpltfODdWHPsRCUpErxgMhJzpeQb2RkXlb-w1GMbc8J-rFEY7X2PzAr0CmWpWhHuIHP%7E48a16P6cYkRMx2t%7ETrGb-WA7xw6paYYSoprXi8n5wQ7W1CmgfJwN7aLps1Ypg8qVyJgcG7DMoVVi1kgCpsj9wVNBEbQUlQm3vCmZzEq10wsDFHiL18XZthx7noWBKmC0NZvFFya5LmzpMHHKXh0ZRI0PVYEHRmWI8sC4Ia2AWRg8ZcyvC4zZr5KtocXawDLY3PQx6KBvT6rjVQahkD%7EXWNCXMzXLi5HRVLQYIAJgx8Cc%7EmTIm2NWVAwMSdo7YZjjmJFM5XQ__&Key-Pair-Id=K15QRJLYKIFSLZ&Download-Request-ID=813015768081005"

        }
    )

Args: {
    "accelerator": "auto",
    "batch_size": 128,
    "callbacks": null,
    "checkpoint": null,
    "data": "/kaggle/working/yolo_dataset/images",
    "devices": "auto",
    "embed_dim": null,
    "epochs": "auto",
    "float32_matmul_precision": "auto",
    "loader_args": null,
    "loggers": null,
    "method": "distillationv1",
    "method_args": {
        "teacher": "dinov3/vits16",
        "teacher_url": "https://dinov3.llamameta.net/dinov3_vits16/dinov3_vits16_pretrain_lvd1689m-08c60483.pth?Policy=eyJTdGF0ZW1lbnQiOlt7InVuaXF1ZV9oYXNoIjoidHZpMngydGx1cjN0eHNtMWd1YWdpYjF3IiwiUmVzb3VyY2UiOiJodHRwczpcL1wvZGlub3YzLmxsYW1hbWV0YS5uZXRcLyoiLCJDb25kaXRpb24iOnsiRGF0ZUxlc3NUaGFuIjp7IkFXUzpFcG9jaFRpbWUiOjE3NTU0OTM0MDZ9fX1dfQ__&Signature=MojlptxwiEPfpltfODdWHPsRCUpErxgMhJzpeQb2RkXlb-w1GMbc8J-rFEY7X2PzAr0CmWpWhHuIHP%7E48a16P6cYkRMx2t%7ETrGb-WA7xw6paYYSoprXi8n5wQ7W1CmgfJwN7aLps1Ypg8qVyJgcG7DMoVVi1kgCpsj9wVNBEbQUlQm3vCmZzEq10wsDFHiL18XZthx7noWBKmC0NZvFFya5LmzpMHHKXh0ZRI0PVYEHRmWI8sC4Ia2A

Downloading: "file:///root/.cache/lightly-train/models/dinov3_vits16_lvd1689m.pth" to /root/.cache/torch/hub/checkpoints/dinov3_vits16_lvd1689m.pth


100%|██████████| 82.5M/82.5M [00:00<00:00, 1.52GB/s]
Resolved configuration:
{
    "accelerator": "CUDAAccelerator",
    "batch_size": 128,
    "callbacks": {
        "device_stats_monitor": null,
        "early_stopping": {
            "check_finite": true,
            "monitor": "train_loss",
            "patience": 1000000000000
        },
        "learning_rate_monitor": {},
        "model_checkpoint": {
            "auto_insert_metric_name": true,
            "dirpath": "/kaggle/working/distilation_RTDTER_results/checkpoints",
            "enable_version_counter": false,
            "every_n_epochs": null,
            "every_n_train_steps": null,
            "filename": null,
            "mode": "min",
            "monitor": null,
            "save_last": true,
            "save_on_train_epoch_end": null,
            "save_top_k": 1,
            "save_weights_only": false,
            "train_time_interval": null,
            "verbose": false
        },
        "model_export": {
  

┏━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name                    ┃ Type                  ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ teacher_embedding_model │ DinoVisionTransformer │ 21.6 M │ eval  │     0 │
│ 1 │ student_embedding_model │ EmbeddingModel        │ 23.5 M │ train │     0 │
│ 2 │ flatten                 │ Flatten               │      0 │ train │     0 │
│ 3 │ student_projection_head │ Linear                │  786 K │ train │     0 │
│ 4 │ criterion               │ DistillationLoss      │      0 │ train │     0 │
└───┴─────────────────────────┴───────────────────────┴────────┴───────┴───────┘

Trainable params: 24.2 M                                                                                           
Non-trainable params: 21.6 M                                                                                       
Total params: 45.9 M                                                                                               
Total estimated model params size (MB): 183                                                                        
Modules in train mode: 277                                                                                         
Modules in eval mode: 212                                                                                          
Total FLOPs: 0

Training: |          | 0/? [00:00<?, ?it/s]

`Trainer.fit` stopped: `max_epochs=100` reached.
Training completed.
Example: How to use the exported model
----------------------------------------------------------------------------------------
import RTDETR # Import the model that was used here
import torch

# Load the pretrained model
model = RTDETR(...)
model.load_state_dict(torch.load('/kaggle/working/distilation_RTDTER_results/exported_models/exported_last.pt', weights_only=True))

# Finetune or evaluate the model
...
----------------------------------------------------------------------------------------

Model exported.


In [ ]:
# from ultralytics import YOLO
# # Load the pretrained model
# model = YOLO('/kaggle/working/distilation_results/exported_models/exported_last.pt')

#finetune of model
# model.train(data="/kaggle/working/yolo_dataset/data.yaml",epochs=25)